# Data Augmentation — Merge AIVIVN 2019 + minhtoan/vietnamese-comment-sentiment
Mục tiêu: bổ sung data đa domain (social media, general comments) vào dataset food review gốc,
giữ nguyên binary label: **0 = negative, 1 = positive**

In [1]:
# ── Setup cho Google Colab ───────────────────────────────────────
# Chạy cell này đầu tiên khi mở trên Colab

import os, sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    # ── Chọn 1 trong 2 cách bên dưới ──

    # CÁCH 1: Project đã có sẵn trên Google Drive
    PROJECT_ROOT = '/content/drive/MyDrive/project_dat391'  # sửa path nếu cần
    os.chdir(PROJECT_ROOT)

    # CÁCH 2: Clone từ GitHub (nếu chưa có trên Drive)
    # !git clone https://github.com/Nhanhaiho/project_dat391.git
    # PROJECT_ROOT = '/content/project_dat391'
    # os.chdir(PROJECT_ROOT)

    print(f'Working directory: {os.getcwd()}')
    print('Files:', os.listdir('.'))
else:
    print('Running locally — không cần mount Drive')

Running locally — không cần mount Drive


In [2]:
import pandas as pd
from datasets import load_dataset

pd.set_option('display.max_colwidth', 80)

c:\Users\Hồ Ngọc Hải Nhân\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load dataset food review gốc

In [3]:
# ── Load data gốc của project ───────────────────────────────────
df_food = pd.read_csv("data/raw/vsa_food_rv_train.csv")

print("=== Food Review (gốc) ===")
print(f"Shape: {df_food.shape}")
print(f"Columns: {df_food.columns.tolist()}")
print(df_food.head(3))
print("\nLabel distribution:")
print(df_food['Rating'].value_counts())

=== Food Review (gốc) ===
Shape: (22847, 2)
Columns: ['Comment', 'Rating']
                                                                           Comment  \
0                                                                    Món ăn cực tệ   
1  Mình thấy mọi người review tích cực nhưng thực sự mình không thấy lẩu ở đây ...   
2  Chắc với sinh viên thì ai cũng quen thuộc với lẩu Nguyệt. Cứ có dịp gì tụ tậ...   

   Rating  
0     0.0  
1     0.0  
2     1.0  

Label distribution:
Rating
1.0    17351
0.0     5490
Name: count, dtype: int64


In [4]:
# Cột thực tế: 'Comment' (text) và 'Rating' (float 0.0/1.0)
df_food = df_food[['Comment', 'Rating']].copy()
df_food.columns = ['review', 'label']
df_food['domain'] = 'food'

# Drop NaN TRƯỚC khi convert — Rating có thể có giá trị NaN
df_food = df_food.dropna(subset=['review', 'label'])
df_food['label'] = df_food['label'].astype(int)  # float 0.0/1.0 → int 0/1

print(f"Food data sau chuẩn hóa: {df_food.shape}")
print(df_food['label'].value_counts())
print(df_food.head(2))

Food data sau chuẩn hóa: (22841, 3)
label
1    17351
0     5490
Name: count, dtype: int64
                                                                            review  \
0                                                                    Món ăn cực tệ   
1  Mình thấy mọi người review tích cực nhưng thực sự mình không thấy lẩu ở đây ...   

   label domain  
0      0   food  
1      0   food  


## 2. Load AIVIVN 2019
Schema thực tế: cột text = `discriptions`, cột label = `mapped_rating` (0=neg, 1=pos)

> Dataset này là **binary** sẵn — không cần drop neutral.

In [5]:
# ── Option A: Load từ HuggingFace ───────────────────────────────
try:
    ds_aivivn = load_dataset("psymbio/aivivn_2019_sentiment")
    df_aivivn_raw = ds_aivivn['train'].to_pandas()
    print("Load từ HuggingFace thành công")
except Exception as e:
    print(f"HuggingFace lỗi: {e}")
    print("→ Dùng Option B: load từ file CSV local")
    # Option B: tải file từ Kaggle rồi để vào data/raw/
    # Link: https://www.kaggle.com/competitions/sentiment-analysis-for-vietnamese
    df_aivivn_raw = pd.read_csv("data/raw/aivivn_train.csv")

print(f"Shape: {df_aivivn_raw.shape}")
print(f"Columns: {df_aivivn_raw.columns.tolist()}")
print(df_aivivn_raw.head(3))
print("\nLabel distribution:")
# Kiểm tra tên cột label thực tế
for col in df_aivivn_raw.columns:
    print(f"  {col}: {df_aivivn_raw[col].unique()[:5]}")

HuggingFace lỗi: Dataset 'psymbio/aivivn_2019_sentiment' doesn't exist on the Hub or cannot be accessed.
→ Dùng Option B: load từ file CSV local
Shape: (12870, 3)
Columns: ['id', 'comment', 'label']
             id  \
0  train_001908   
1  train_006244   
2  train_015521   

                                                                           comment  \
0                                   Shop phục vụ rất kémKo có đạo đức nghề nghiệp    
1            sản phẩm giống hình nhưng áo rộng dài còn quần ngắn không ưng ý laems   
2   Chất lượng sản phẩm tuyệt vời Chất lượng sản phẩm tuyệt vời Đóng gói sản ph...   

   label  
0      0  
1      0  
2      1  

Label distribution:
  id: ['train_001908' 'train_006244' 'train_015521' 'train_012157'
 'train_002251']
  comment: [' Shop phục vụ rất kémKo có đạo đức nghề nghiệp\xa0'
 'sản phẩm giống hình nhưng áo rộng dài còn quần ngắn không ưng ý laems'
 ' Chất lượng sản phẩm tuyệt vời Chất lượng sản phẩm tuyệt vời Đóng gói sản phẩm rất đẹp và 

In [6]:
# ── Chuẩn hóa AIVIVN ────────────────────────────────────────────
# Tên cột thực tế theo schema đã xác nhận từ paper nghiên cứu
AIVIVN_TEXT  = 'discriptions'   # hoặc 'sentence' tùy mirror trên HF
AIVIVN_LABEL = 'mapped_rating'  # hoặc 'label' tùy mirror trên HF

# Nếu HuggingFace mirror dùng tên khác, auto-detect:
text_candidates  = ['discriptions', 'sentence', 'text', 'comment', 'review']
label_candidates = ['mapped_rating', 'label', 'sentiment', 'rating']

actual_text  = next((c for c in text_candidates  if c in df_aivivn_raw.columns), None)
actual_label = next((c for c in label_candidates if c in df_aivivn_raw.columns), None)

print(f"Detected → text col: '{actual_text}', label col: '{actual_label}'")

df_aivivn = df_aivivn_raw[[actual_text, actual_label]].copy()
df_aivivn.columns = ['review', 'label']
df_aivivn['label'] = df_aivivn['label'].astype(int)
df_aivivn['domain'] = 'social_general'

# AIVIVN là binary rồi — chỉ cần drop NaN
df_aivivn = df_aivivn.dropna(subset=['review', 'label'])

print(f"\nAIVIVN sau chuẩn hóa: {df_aivivn.shape}")
print(df_aivivn['label'].value_counts())
print(df_aivivn.head(3))

Detected → text col: 'comment', label col: 'label'

AIVIVN sau chuẩn hóa: (12870, 3)
label
1    7424
0    5446
Name: count, dtype: int64
                                                                            review  \
0                                   Shop phục vụ rất kémKo có đạo đức nghề nghiệp    
1            sản phẩm giống hình nhưng áo rộng dài còn quần ngắn không ưng ý laems   
2   Chất lượng sản phẩm tuyệt vời Chất lượng sản phẩm tuyệt vời Đóng gói sản ph...   

   label          domain  
0      0  social_general  
1      0  social_general  
2      1  social_general  


## 3. Load minhtoan/vietnamese-comment-sentiment
Schema thực tế: cột text = `Content`, cột label = `Sentiment` (positive/negative/**neutral**)

> Dataset này có **3 class** — phải **drop neutral** để giữ binary.

In [7]:
# ── Load minhtoan ───────────────────────────────────────────────
try:
    ds_social = load_dataset("minhtoan/vietnamese-comment-sentiment")
    df_social_raw = ds_social['train'].to_pandas()
    print("Load từ HuggingFace thành công")
except Exception as e:
    print(f"Lỗi: {e}")

print(f"Shape: {df_social_raw.shape}")
print(f"Columns: {df_social_raw.columns.tolist()}")
print("\nSentiment distribution:")
print(df_social_raw['Sentiment'].value_counts())
print(df_social_raw[['Content', 'Sentiment']].head(3))

Load từ HuggingFace thành công
Shape: (13132, 16)
Columns: ['ID', 'Title', 'Content', 'BriefContent', 'URL', 'Published Date', 'Week', 'Keyword', 'Group', 'Sub', 'Keyword 2', 'Sentiment', 'Ngành', 'Source', 'Channel', 'Author']

Sentiment distribution:
Sentiment
Trung lập    6968
Tích cực     2838
neutral      2527
positive      575
Tiêu cực      192
negative       32
Name: count, dtype: int64
                                                                           Content  \
0  -  NĐTNN mua ròng trở  lại trên sàn HOSE về  mặt KLGD sau 2 phiên bán ròng ,...   
1  -  NĐTNN mua ròng trở  lại trên sàn HOSE về  mặt KLGD sau 2 phiên bán ròng ,...   
2  -  NĐTNN mua ròng trở  lại trên sàn HOSE về  mặt KLGD sau 2 phiên bán ròng ,...   

   Sentiment  
0  Trung lập  
1  Trung lập  
2  Trung lập  


In [8]:
# ── Chuẩn hóa minhtoan ─────────────────────────────────────────
df_social = df_social_raw[['Content', 'Sentiment']].copy()
df_social.columns = ['review', 'sentiment_str']

# Kiểm tra giá trị label thực tế (có thể viết hoa/thường khác nhau)
print("Unique sentiment values:", df_social['sentiment_str'].unique())

# Normalize về lowercase để xử lý nhất quán
df_social['sentiment_str'] = df_social['sentiment_str'].str.lower().str.strip()

# !! QUAN TRỌNG: Drop neutral — giữ binary pos/neg
before = len(df_social)
df_social = df_social[df_social['sentiment_str'].isin(['positive', 'negative'])].copy()
after = len(df_social)
print(f"\nDrop neutral: {before} → {after} rows ({before - after} neutral bị bỏ)")

# Map string → int
df_social['label'] = df_social['sentiment_str'].map({'positive': 1, 'negative': 0})
df_social['domain'] = 'social_media'
df_social = df_social[['review', 'label', 'domain']]

print(f"\nminhtoan sau chuẩn hóa: {df_social.shape}")
print(df_social['label'].value_counts())
print(df_social.head(3))

Unique sentiment values: ['Trung lập' 'Tích cực' 'Tiêu cực' 'neutral' 'positive' 'negative']

Drop neutral: 13132 → 607 rows (12525 neutral bị bỏ)

minhtoan sau chuẩn hóa: (607, 3)
label
1    575
0     32
Name: count, dtype: int64
                                                                                review  \
10011      Hơn 1 tuôi thi uông thêm sưa tươi cucng đc, be nha m hay uông vinamilk 100%   
10012  Trong một tháng qua kể từ khi Chính phủ tuyên bố thoái vốn khỏi 10 doanh ngh...   
10165                                             E uống vinamilk k đường loại 100% ấy   

       label        domain  
10011      1  social_media  
10012      1  social_media  
10165      1  social_media  


## 4. Merge tất cả + làm sạch

In [9]:
import re

def clean_text(text):
    """Làm sạch nhẹ — giữ dấu tiếng Việt, emoji, slang"""
    text = str(text).strip()
    # Bỏ URL
    text = re.sub(r'http\S+|www\.\S+', '', text)
    # Chuẩn hóa khoảng trắng
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Merge
df_merged = pd.concat([
    df_food,
    df_aivivn,
    df_social,
], ignore_index=True)

print(f"Sau merge (trước clean): {df_merged.shape}")

# Làm sạch text
df_merged['review'] = df_merged['review'].apply(clean_text)

# Loại bỏ NaN
df_merged = df_merged.dropna(subset=['review', 'label'])

# Loại review quá ngắn (< 5 ký tự)
df_merged = df_merged[df_merged['review'].str.len() >= 5]

# Loại duplicate theo nội dung review
before_dedup = len(df_merged)
df_merged = df_merged.drop_duplicates(subset=['review'])
after_dedup = len(df_merged)

# Đảm bảo label là int
df_merged['label'] = df_merged['label'].astype(int)

print(f"Sau drop duplicate: {before_dedup} → {after_dedup} rows")

Sau merge (trước clean): (36318, 3)
Sau drop duplicate: 36239 → 26214 rows


In [10]:
# ── Thống kê tổng thể ───────────────────────────────────────────
print("="*50)
print("THỐNG KÊ DATASET SAU MERGE")
print("="*50)
print(f"\nTổng samples: {len(df_merged):,}")

print("\nTheo domain:")
domain_stats = df_merged.groupby('domain').agg(
    count=('review', 'count'),
    pos=('label', 'sum')
)
domain_stats['neg'] = domain_stats['count'] - domain_stats['pos']
domain_stats['pos_pct'] = (domain_stats['pos'] / domain_stats['count'] * 100).round(1)
print(domain_stats)

print("\nTheo label (tổng):")
print(df_merged['label'].value_counts())
total = len(df_merged)
pos_total = df_merged['label'].sum()
print(f"  Pos ratio: {pos_total/total*100:.1f}% / Neg ratio: {(total-pos_total)/total*100:.1f}%")

THỐNG KÊ DATASET SAU MERGE

Tổng samples: 26,214

Theo domain:
                count   pos   neg  pos_pct
domain                                    
food            12906  9624  3282     74.6
social_general  12809  7381  5428     57.6
social_media      499   469    30     94.0

Theo label (tổng):
label
1    17474
0     8740
Name: count, dtype: int64
  Pos ratio: 66.7% / Neg ratio: 33.3%


## 5. Tính class weight — giữ toàn bộ 26K samples
Thay vì undersample (mất ~8K rows), ta tính `class_weight` để truyền vào `WeightedTrainer`.
Model vẫn học balanced mà không lãng phí data.

In [11]:
import numpy as np
import torch

# Giữ toàn bộ data — không undersample
df_balanced = df_merged.copy()

pos_count = (df_balanced['label'] == 1).sum()
neg_count = (df_balanced['label'] == 0).sum()
total     = len(df_balanced)

print(f"Pos: {pos_count:,} | Neg: {neg_count:,} | Total: {total:,}")
print(f"Ratio pos/neg: {pos_count/neg_count:.2f}")

# Tính class weight: w_i = total / (n_classes * count_i)
# → class thiểu số được weight cao hơn
weight_neg = total / (2 * neg_count)
weight_pos = total / (2 * pos_count)
class_weights = torch.tensor([weight_neg, weight_pos], dtype=torch.float)

print(f"\nClass weights:")
print(f"  negative (0): {weight_neg:.4f}")
print(f"  positive (1): {weight_pos:.4f}")
print("\n→ Truyền class_weights vào WeightedTrainer khi fine-tune PhoBERT (xem cell tiếp theo)")

Pos: 17,474 | Neg: 8,740 | Total: 26,214
Ratio pos/neg: 2.00

Class weights:
  negative (0): 1.4997
  positive (1): 0.7501

→ Truyền class_weights vào WeightedTrainer khi fine-tune PhoBERT (xem cell tiếp theo)


## 6. Split train/val/test và lưu

In [12]:
from sklearn.model_selection import train_test_split

# Shuffle toàn bộ
df_final = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

# Split: 80% train, 10% val, 10% test  (stratify theo label)
df_train, df_temp = train_test_split(
    df_final, test_size=0.2, random_state=42, stratify=df_final['label']
)
df_val, df_test = train_test_split(
    df_temp, test_size=0.5, random_state=42, stratify=df_temp['label']
)

print(f"Train: {len(df_train):,} | Val: {len(df_val):,} | Test: {len(df_test):,}")

# Lưu splits (2 cột review + label cho training)
for df_split, name in [(df_train, 'train_split'), (df_val, 'val_split'), (df_test, 'test_split_from_train')]:
    out_path = f"data/processed/{name}.csv"
    df_split[['review', 'label']].to_csv(out_path, index=False, encoding='utf-8-sig')
    print(f"Saved: {out_path} ({len(df_split):,} rows)")

# Lưu full dataset có domain tag
df_final.to_csv("data/raw/vsa_general_sentiment_full.csv", index=False, encoding='utf-8-sig')

# Lưu class_weights để dùng trong WeightedTrainer
torch.save(class_weights, "data/processed/class_weights.pt")
print("\n✓ Saved: data/raw/vsa_general_sentiment_full.csv")
print("✓ Saved: data/processed/class_weights.pt")

Train: 20,971 | Val: 2,621 | Test: 2,622
Saved: data/processed/train_split.csv (20,971 rows)
Saved: data/processed/val_split.csv (2,621 rows)
Saved: data/processed/test_split_from_train.csv (2,622 rows)

✓ Saved: data/raw/vsa_general_sentiment_full.csv
✓ Saved: data/processed/class_weights.pt


## 7. WeightedTrainer — dùng thay Trainer khi fine-tune PhoBERT
Copy class này vào `src/phobert_pipeline.py` hoặc notebook training của bạn.

In [13]:
# ── WeightedTrainer: thay thế Trainer gốc ───────────────────────
# Dùng class_weights đã tính ở trên để xử lý imbalanced data
# mà không cần undersample hay bỏ data

from transformers import Trainer
from torch.nn import CrossEntropyLoss

class WeightedTrainer(Trainer):
    """
    Custom Trainer tính loss có trọng số theo class.
    Negative (class 0) được weight cao hơn vì ít samples hơn.
    """
    def __init__(self, class_weights, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits  = outputs.logits

        # Đưa weight lên cùng device với model
        weights = self.class_weights.to(logits.device)
        loss = CrossEntropyLoss(weight=weights)(logits, labels)

        return (loss, outputs) if return_outputs else loss


# ── Cách dùng trong notebook training (03_phobert_training.ipynb) ─
# Thay dòng:
#   trainer = build_trainer(train_tok, val_tok, tokenizer, output_dir=...)
# Thành:
#
# class_weights = torch.load("data/processed/class_weights.pt")
#
# trainer = WeightedTrainer(
#     class_weights = class_weights,
#     model         = model,
#     args          = training_args,
#     train_dataset = train_tok,
#     eval_dataset  = val_tok,
#     compute_metrics = compute_metrics,
# )
# trainer.train()

print("WeightedTrainer định nghĩa xong.")
print(f"Class weights: neg={class_weights[0]:.4f}, pos={class_weights[1]:.4f}")

WeightedTrainer định nghĩa xong.
Class weights: neg=1.4997, pos=0.7501


In [14]:
# ── Kiểm tra nhanh ──────────────────────────────────────────────
print("=== Sample từ mỗi domain ===")
for domain in df_final['domain'].unique():
    sample = df_final[df_final['domain'] == domain].sample(1)['review'].values[0]
    print(f"\n[{domain}]")
    print(f"  {sample[:120]}")

=== Sample từ mỗi domain ===

[food]
  Quán nằm trên 31 phố Hàng Chuối, quán cũng khá đông khách. Quán có bún, miến, bánh đa riêu cua, bò, giò,đậu, chả cá. Mìn

[social_general]
  Mua online nhưng rất tuyệt vời mặt 32 mm cho nữ đeo rất vừa vặn tks shop nhiều :)

[social_media]
  Vơi bât cư môt doanh nghiêp nao, vai tro cua ban lanh đao đong vai tro sông con tơi sư phat triên cung như gia tri cua d


In [15]:
# ── Lưu kết quả về Google Drive (chỉ cần nếu dùng Colab) ────────
if IN_COLAB:
    import shutil
    # Các file đã được lưu thẳng vào PROJECT_ROOT/data/processed/
    # nên đã có trên Drive rồi — chỉ cần verify
    for f in ['train_split.csv', 'val_split.csv',
              'test_split_from_train.csv', 'class_weights.pt']:
        fpath = f'data/processed/{f}'
        exists = os.path.exists(fpath)
        print(f'  {fpath}: {"✓ OK" if exists else "✗ MISSING"}')
    print('\n✓ Tất cả file đã lưu vào Drive tại:', PROJECT_ROOT)